# Exploração dos Dados

## Importação de Bibliotecas Necessárias

In [16]:
# Importar pacotes

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder,  MinMaxScaler
from sklearn.compose import ColumnTransformer

## Importação das Planilhas de Dados

In [17]:
# Carregar os arquivos CSV

# Carregamento direto e completo

# df_gf = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GF.csv", sep="\t", header=0, engine="python")
# df_gl = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GL.csv", sep="\t", header=0, engine="python")
# df_gm = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GM.csv", sep="\t", header=0, engine="python")
# df_gp = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GP.csv", sep="\t", header=0, engine="python")
# df_gt = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GT.csv", sep="\t", header=0, engine="python")
# df_gv = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GV.csv", sep="\t", header=0, engine="python")
# df_gw = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GW.csv", sep="\t", header=0, engine="python")

# Carregamento em amostras para economia de memória e testes

df_gf = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GF.csv", sep="\t", header=0, engine="python").sample(n=5000, random_state=42)
df_gl = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GL.csv", sep="\t", header=0, engine="python").sample(n=5000, random_state=42)
df_gm = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GM.csv", sep="\t", header=0, engine="python").sample(n=5000, random_state=42)
df_gp = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GP.csv", sep="\t", header=0, engine="python").sample(n=5000, random_state=42)
df_gt = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GT.csv", sep="\t", header=0, engine="python").sample(n=5000, random_state=42)
df_gv = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GV.csv", sep="\t", header=0, engine="python").sample(n=5000, random_state=42)
df_gw = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GW.csv", sep="\t", header=0, engine="python").sample(n=5000, random_state=42)

# Após carregar, conversão de floats para float32 para economizar memória também

for df in [df_gf, df_gl, df_gm, df_gp, df_gt, df_gv, df_gw]:
    for col in df.select_dtypes(include="float64").columns:
        df[col] = df[col].astype("float32")


 ## Divisão de Colunas em Múltiplas Colunas

In [18]:
# Função para a divisão das colunas

def dividir_colunas(df):

    # Verifica se o DataFrame tem apenas uma coluna.

    if df.shape[1] == 1:

        # Divide a única coluna em múltiplas colunas.

        dados = df.iloc[:, 0].str.split('\t', expand=True)

        # Usa a primeira linha como nomes das colunas.

        dados.columns = dados.iloc[0]

        # Remove a primeira linha por ter virado cabeçalho.

        dados = dados.drop(index=0).reset_index(drop=True)
        return dados
    else:

        return df

df_gf = dividir_colunas(df_gf)
df_gl = dividir_colunas(df_gl)
df_gm = dividir_colunas(df_gm)
df_gp = dividir_colunas(df_gp)
df_gt = dividir_colunas(df_gt)
df_gv = dividir_colunas(df_gv)
df_gw = dividir_colunas(df_gw)

## Conversão de Tipos de Dados

In [19]:
# Divisão de tipos

colunas_numericas = ['id_grupo', 'id_pagador', 'id_beneficiario', 'Numero_do_boleto', 'vl_boleto', 'vl_pagto', 'juros']
colunas_categoricas = ['status_boleto', 'pagador_cep', 'banco', 'tipo_juros', 'pagador_cidade']
colunas_data = ['data_inclusao', 'data_vencto', 'dt_pagto']
colunas_texto = [ 'pagador_cep', 'pagador_cnpjcpf', 'pagador_inscricao_hash']

# Função para converter em um DataFrame

def converter_tipos(df):

    # Converte colunas numéricas para tipo numérico, formatando '\N' como NaN

    for col in colunas_numericas:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].replace('\\N', np.nan), errors='coerce')

    # Converte colunas categóricas para category

    for col in colunas_categoricas:
        if col in df.columns:
            df[col] = df[col].astype('category')

    # Converte colunas de data para tipo datetime, formatando '\N' como NaN

    for col in colunas_data:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col].replace('\\N', np.nan), errors='coerce')

# Aplica a função de conversão de tipos para cada DataFrame

for df in [df_gf, df_gl, df_gm, df_gp, df_gt, df_gv, df_gw]:
    converter_tipos(df)

## Tratamento de Valores Ausentes

In [20]:
# Colunas para remoção considerando que tem muitos valores ausentes

colunas_para_remover = ['tipo_multa', 'multa']

for df in [df_gf, df_gl, df_gm, df_gp, df_gt, df_gv, df_gw]:

    # Remove colunas desnecessárias

    df.drop(columns=[c for c in colunas_para_remover if c in df.columns], inplace=True)

    # Substitui valores ausentes em vl_pagto por 0

    if 'vl_pagto' in df.columns:
        df['vl_pagto'] = df['vl_pagto'].fillna(0)

    # Trata valores ausentes em pagador_cidade

    if 'pagador_cidade' in df.columns:
        df['pagador_cidade'] = df['pagador_cidade'].cat.add_categories('Desconhecido')
        df['pagador_cidade'] = df['pagador_cidade'].fillna('Desconhecido')


## Identificação e Tratamento de Outliers

In [21]:
# Verificação em quartis

for df in [df_gf, df_gl, df_gm, df_gp, df_gt, df_gv, df_gw]:
    for col in colunas_numericas:
        if col in df.columns:

            # Calcula o primeiro quartil (Q1)

            Q1 = df[col].quantile(0.25)

            # Calcula o terceiro quartil (Q3)

            Q3 = df[col].quantile(0.75)

            # Calcula o Intervalo Interquartil (IQR)

            IQR = Q3 - Q1

            # Define os limites inferior e superior para identificar os outliers

            lim_inf = Q1 - 1.5 * IQR
            lim_sup = Q3 + 1.5 * IQR

            # Substitui valores fora dos limites pelos próprios limites

            df[col] = np.clip(df[col], lim_inf, lim_sup)

In [22]:
print(df_gf.columns.tolist())


['id_grupo', 'id_beneficiario', 'Numero_do_boleto', 'data_inclusao', 'status_boleto', 'data_vencto', 'vl_boleto', 'dt_pagto', 'vl_pagto', 'banco', 'id_pagador', 'pagador_cep', 'pagador_cidade', 'qtd_acessos_pagador', 'pagador_dt_ultimo_acesso', 'pagador_cnpjcpf', 'pagador_inscricao_hash', 'valor_abatimento', 'tipo_juros', 'juros']


## Tratamento de Datas


In [23]:
# Função para extrair os elementos de data (ano, mês, dia) e remover colunas originais

def tratar_datas(df):

    df_dates = df.copy()

    # Itera sobre a lista de colunas de data

    for col in colunas_data:

        if col in df_dates.columns:

            # Converte a coluna para datetime, tratando os erros com coerce

            df_dates[col] = pd.to_datetime(df_dates[col], errors='coerce')

            # Extrai o ano e cria uma nova coluna

            df_dates[f"{col}_ano"] = df_dates[col].dt.year

            # Extrai o mês e cria uma nova coluna

            df_dates[f"{col}_mes"] = df_dates[col].dt.month

            # Extrai o dia e cria uma nova coluna

            df_dates[f"{col}_dia"] = df_dates[col].dt.day

    # Remove as colunas de data originais
    return df_dates.drop(columns=colunas_data, errors="ignore")

## Pré-processamento (Normalização e Codificação)

In [24]:
# Pré-processamento de datas


colunas_numericas = ['id_grupo','id_pagador','id_beneficiario','Numero_do_boleto',
                     'vl_boleto','vl_pagto','juros','multa','valor_abatimento','qtd_acessos_pagador']
colunas_categoricas = ['status_boleto','banco','pagador_cidade','tipo_juros','tipo_multa']
colunas_data = ['data_inclusao','data_vencto','dt_pagto','pagador_dt_ultimo_acesso']
colunas_texto = ['pagador_cep','pagador_cnpjcpf','pagador_inscricao_hash']

colunas_nao_normalizar = ['id_grupo','id_beneficiario','Numero_do_boleto','id_pagador',
                          'pagador_cep','data_inclusao','data_vencto','dt_pagto','pagador_dt_ultimo_acesso']

def tratar_datas(df):
    df_out = df.copy()
    for col in colunas_data:
        if col in df_out.columns:

            # Força o formato dia/mês/ano

            df_out[col] = pd.to_datetime(df_out[col], errors='coerce', format='%d/%m/%Y')
            df_out[f'{col}_ano'] = df_out[col].dt.year
            df_out[f'{col}_mes'] = df_out[col].dt.month
            df_out[f'{col}_dia'] = df_out[col].dt.day
    return df_out


In [25]:
# Pré-processamento (One-Hot e Normalização no escopo de 0 a 10000)

def preprocessar(df):
    df_proc = tratar_datas(df)

    # One-Hot nas categóricas

    cat_cols = [c for c in colunas_categoricas if c in df_proc.columns]
    if cat_cols:
        enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        enc_arr = enc.fit_transform(df_proc[cat_cols])
        enc_cols = enc.get_feature_names_out(cat_cols)
        df_ohe = pd.DataFrame(enc_arr, columns=enc_cols, index=df_proc.index)
        df_proc = pd.concat([df_proc.drop(columns=cat_cols), df_ohe], axis=1)

    # Normalização numérica

    num_candidatas = [c for c in colunas_numericas if c in df_proc.columns]
    num_para_normalizar = [c for c in num_candidatas if c not in colunas_nao_normalizar]

    if num_para_normalizar:
        X_num = df_proc[num_para_normalizar].copy()

        # Substitui '\N' por NaN

        X_num = X_num.replace('\\N', np.nan)

        # Converte para numérico

        X_num = X_num.apply(pd.to_numeric, errors='coerce')

        # Preenche NaN com mediana

        X_num = X_num.fillna(X_num.median(numeric_only=True))

        scaler = MinMaxScaler(feature_range=(0, 10000))
        arr_norm = scaler.fit_transform(X_num)
        df_norm = pd.DataFrame(arr_norm, columns=[f'{c}_norm' for c in num_para_normalizar], index=df_proc.index)
        for c in df_norm.columns:
            df_norm[c] = df_norm[c].astype('float32')
        df_proc = pd.concat([df_proc, df_norm], axis=1)

    # Conversão float64 pra float32

    for c in df_proc.select_dtypes(include='float64').columns:
        df_proc[c] = df_proc[c].astype('float32')

    # Adiciona a coluna pagador_cidade original de volta

    if 'pagador_cidade' in df.columns:
        df_proc['pagador_cidade_original'] = df['pagador_cidade']


    return df_proc

 ## Aplicar Pré-processamento

In [26]:

df_gf_pre = preprocessar(df_gf)
df_gl_pre = preprocessar(df_gl)
df_gm_pre = preprocessar(df_gm)
df_gp_pre = preprocessar(df_gp)
df_gt_pre = preprocessar(df_gt)
df_gv_pre = preprocessar(df_gv)
df_gw_pre = preprocessar(df_gw)

/tmp/ipython-input-3302719441.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_proc['pagador_cidade_original'] = df['pagador_cidade']
/tmp/ipython-input-3302719441.py:26: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_num = X_num.replace('\\N', np.nan)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/_array_api.py:776: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
/usr/local/lib/python3.12/dist-packages/sklearn/utils/_array_api.py:793: RuntimeWarning: All-NaN slice encounte

 ## Visualização Prévia

In [27]:
# Iteração para visualizar resultados

for nome, df_proc in {"GF": df_gf_pre, "GL": df_gl_pre, "GM": df_gm_pre, "GP": df_gp_pre, "GT": df_gt_pre, "GV": df_gv_pre, "GW": df_gw_pre}.items():
    print(f"\n {nome}")
    display(df_proc.head(3))
    print(df_proc.info())


 GF


,id_grupo,id_beneficiario,Numero_do_boleto,data_inclusao,data_vencto,vl_boleto,dt_pagto,vl_pagto,id_pagador,pagador_cep,...,pagador_cidade_SOROCABA,pagador_cidade_SUZANO,tipo_juros_M,tipo_juros_S,vl_boleto_norm,vl_pagto_norm,juros_norm,valor_abatimento_norm,qtd_acessos_pagador_norm,pagador_cidade_original
14278,41,85,380162593,2024-07-03,2024-07-22,329.329987,NaT,0.00000,14338604,9171200,...,0.0,0.0,1.0,0.0,5568.630859,0.000000,6250.000000,0.0,526.315796,SANTO ANDRE (SP
16224,41,111,420644273,2024-10-30,2024-11-20,283.420013,NaT,0.00000,18839668,3220300,...,0.0,0.0,1.0,0.0,4055.248291,0.000000,3750.000000,0.0,526.315796,SAO PAULO
7646,41,75,430304903,2024-12-11,2025-01-22,376.000000,2025-02-03,384.51001,20113965,4304010,...,0.0,0.0,1.0,0.0,7107.067383,4843.732422,7499.999512,0.0,526.315796,SAO PAULO


<class 'pandas.core.frame.DataFrame'>
Index: 5000 entries, 14278 to 8576
Data columns (total 63 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   id_grupo                             5000 non-null   int64         
 1   id_beneficiario                      5000 non-null   int64         
 2   Numero_do_boleto                     5000 non-null   int64         
 3   data_inclusao                        5000 non-null   datetime64[ns]
 4   data_vencto                          5000 non-null   datetime64[ns]
 5   vl_boleto                            5000 non-null   float32       
 6   dt_pagto                             3479 non-null   datetime64[ns]
 7   vl_pagto                             5000 non-null   float32       
 8   id_pagador                           5000 non-null   int64         
 9   pagador_cep                          5000 non-null   category      
 10  qtd_acessos_p

,id_grupo,id_beneficiario,Numero_do_boleto,data_inclusao,data_vencto,vl_boleto,dt_pagto,vl_pagto,id_pagador,pagador_cep,...,pagador_cidade_VILA VELHA,pagador_cidade_VITORIA,pagador_cidade_VOLTA REDONDA,tipo_juros_M,vl_boleto_norm,vl_pagto_norm,juros_norm,valor_abatimento_norm,qtd_acessos_pagador_norm,pagador_cidade_original
4463,173,554,424652181,2024-11-18,2024-12-19,605.700012,2024-12-20,605.700012,2669816,79004060,...,0.0,0.0,0.0,1.0,217.104507,264.965179,220.264328,0.0,303.030304,CAMPO GRANDE
4818,173,554,427422104,2024-11-28,2025-01-09,2896.000000,2025-01-10,2896.000000,3388364,83833072,...,0.0,0.0,0.0,1.0,1057.680664,1266.863525,1068.281982,0.0,303.030304,FAZENDA RIO GRA
7298,173,554,452659794,2025-03-20,2025-04-18,27260.947266,2025-04-23,22859.605469,2672233,14085070,...,0.0,0.0,0.0,1.0,10000.000000,10000.000000,10000.000000,0.0,303.030304,RIBEIRAO PRETO


<class 'pandas.core.frame.DataFrame'>
Index: 5000 entries, 4463 to 191
Columns: 172 entries, id_grupo to pagador_cidade_original
dtypes: category(2), datetime64[ns](4), float32(153), int32(6), int64(4), object(3)
memory usage: 3.5+ MB
None

 GM


...

<class 'pandas.core.frame.DataFrame'>
Index: 5000 entries, 371815 to 29933
Columns: 562 entries, id_grupo to pagador_cidade_original
dtypes: category(2), datetime64[ns](4), float32(544), int32(6), int64(3), object(3)
memory usage: 11.1+ MB
None

 GV


,id_grupo,id_beneficiario,Numero_do_boleto,data_inclusao,data_vencto,vl_boleto,dt_pagto,vl_pagto,id_pagador,pagador_cep,...,status_boleto_REGISTRADO,banco_BANCO BRADESCO SA,pagador_cidade_Desconhecido,tipo_juros_P,vl_boleto_norm,vl_pagto_norm,juros_norm,valor_abatimento_norm,qtd_acessos_pagador_norm,pagador_cidade_original
189406,67,213,453755204,2025-03-24,2025-04-18,203.000000,2025-04-23,203.000000,4241566,83327170,...,0.0,1.0,1.0,1.0,1084.449951,1252.274780,0.0,0.0,983.374573,Desconhecido
192861,67,213,454450324,2025-03-26,2025-04-29,359.000000,2025-04-30,359.000000,1137472,89520000,...,0.0,1.0,1.0,1.0,2056.715576,2214.614014,0.0,0.0,3.537319,Desconhecido
230619,67,213,464215356,2025-05-08,2025-06-11,356.899994,2025-06-12,356.899994,416835,99260000,...,0.0,1.0,1.0,1.0,2043.627197,2201.659424,0.0,0.0,0.000000,Desconhecido


<class 'pandas.core.frame.DataFrame'>
Index: 5000 entries, 189406 to 28111
Data columns (total 41 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   id_grupo                      5000 non-null   int64         
 1   id_beneficiario               5000 non-null   int64         
 2   Numero_do_boleto              5000 non-null   int64         
 3   data_inclusao                 5000 non-null   datetime64[ns]
 4   data_vencto                   5000 non-null   datetime64[ns]
 5   vl_boleto                     5000 non-null   float32       
 6   dt_pagto                      4648 non-null   datetime64[ns]
 7   vl_pagto                      5000 non-null   float32       
 8   id_pagador                    5000 non-null   int64         
 9   pagador_cep                   5000 non-null   category      
 10  qtd_acessos_pagador           5000 non-null   object        
 11  pagador_dt_ultimo_acesso     

,id_grupo,id_beneficiario,Numero_do_boleto,data_inclusao,data_vencto,vl_boleto,dt_pagto,vl_pagto,id_pagador,pagador_cep,...,pagador_cidade_Xaxim,pagador_cidade_Xique-Xique,pagador_cidade_ilheus,tipo_juros_M,vl_boleto_norm,vl_pagto_norm,juros_norm,valor_abatimento_norm,qtd_acessos_pagador_norm,pagador_cidade_original
2104,212,658,384398448,2024-07-15,2024-12-09,1351.500000,2024-12-10,1351.50000,3255052,64202020,...,0.0,0.0,0.0,1.0,869.366699,2492.493408,871.248779,0.0,1111.111084,PARNAIBA
75425,212,658,445781813,2025-02-18,2025-05-19,6318.600098,NaT,0.00000,3257350,13202571,...,0.0,0.0,0.0,1.0,4076.109131,0.000000,4085.188721,0.0,0.000000,JUNDIAI
18926,212,658,399294026,2024-08-22,2024-12-17,6300.000000,NaT,5422.28125,3889099,59124400,...,0.0,0.0,0.0,1.0,4064.100830,10000.000000,4065.827637,0.0,0.000000,Natal


<class 'pandas.core.frame.DataFrame'>
Index: 5000 entries, 2104 to 110411
Columns: 782 entries, id_grupo to pagador_cidade_original
dtypes: category(2), datetime64[ns](4), float32(763), int32(6), int64(4), object(3)
memory usage: 15.2+ MB
None


 ## Estatística Descritiva


In [28]:
# Método describe aplicado

for nome, df_proc in {"GF": df_gf_pre, "GL": df_gl_pre, "GM": df_gm_pre, "GP": df_gp_pre, "GT": df_gt_pre, "GV": df_gv_pre, "GW": df_gw_pre}.items():
    print(f"\n Estatísticas {nome}")
    display(df_proc.describe())


 Estatísticas GF


,id_grupo,id_beneficiario,Numero_do_boleto,data_inclusao,data_vencto,vl_boleto,dt_pagto,vl_pagto,id_pagador,pagador_dt_ultimo_acesso,...,pagador_cidade_SAO PAULO,pagador_cidade_SOROCABA,pagador_cidade_SUZANO,tipo_juros_M,tipo_juros_S,vl_boleto_norm,vl_pagto_norm,juros_norm,valor_abatimento_norm,qtd_acessos_pagador_norm
count,5000.0,5000.000000,5.000000e+03,5000,5000,5000.000000,3479,5000.000000,5.000000e+03,0,...,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.0,5000.000000
mean,41.0,99.009200,4.322851e+08,2024-12-27 19:08:32.640000,2025-01-21 07:23:48.480000,311.661591,2025-01-16 23:43:51.445817856,214.662781,1.092486e+07,NaT,...,0.507600,0.000200,0.000200,0.999600,0.000400,4986.208008,2704.140381,5414.250000,0.0,570.105286
min,41.0,61.000000,3.801625e+08,2024-07-03 00:00:00,2024-07-12 00:00:00,160.400009,2024-07-04 00:00:00,0.000000,3.335300e+04,NaT,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
25%,41.0,85.000000,4.125212e+08,2024-09-27 00:00:00,2024-10-22 00:00:00,274.160004,2024-10-11 00:00:00,0.000000,2.971219e+06,NaT,...,0.000000,0.000000,0.000000,1.000000,0.000000,3750.000000,0.000000,3750.000000,0.0,526.315796
50%,41.0,97.000000,4.297290e+08,2024-12-09 00:00:00,2025-01-20 00:00:00,314.290009,2025-01-09 00:00:00,273.664993,7.034871e+06,NaT,...,1.000000,0.000000,0.000000,1.000000,0.000000,5072.851074,3447.400635,5000.000488,0.0,526.315796
75%,41.0,119.000000,4.553448e+08,2025-03-31 00:00:00,2025-04-22 00:00:00,350.000000,2025-04-24 00:00:00,336.714996,1.926927e+07,NaT,...,1.000000,0.000000,0.000000,1.000000,0.000000,6250.000000,4241.651367,6250.000000,0.0,526.315796
max,41.0,129.000000,4.763794e+08,2025-06-30 00:00:00,2025-07-30 00:00:00,463.760010,2025-08-12 00:00:00,793.830017,2.113007e+07,NaT,...,1.000000,1.000000,1.000000,1.000000,1.000000,10000.000000,10000.000000,10000.000000,0.0,10000.000000
std,0.0,20.314725,2.766280e+07,NaN,NaN,60.973232,NaN,151.099640,8.072541e+06,NaN,...,0.499982,0.014142,0.014142,0.019998,0.019998,2009.929443,1903.431030,2296.946289,0.0,406.322357



 Estatísticas GL


,id_grupo,id_beneficiario,Numero_do_boleto,data_inclusao,data_vencto,vl_boleto,dt_pagto,vl_pagto,id_pagador,pagador_dt_ultimo_acesso,...,pagador_cidade_VIANOPOLIS,pagador_cidade_VILA VELHA,pagador_cidade_VITORIA,pagador_cidade_VOLTA REDONDA,tipo_juros_M,vl_boleto_norm,vl_pagto_norm,juros_norm,valor_abatimento_norm,qtd_acessos_pagador_norm
count,5000.0,5000.0,5.000000e+03,5000,5000,5000.000000,4500,5000.000000,5.000000e+03,0,...,5000.000000,5000.00000,5000.000000,5000.000000,5000.0,5000.000000,5000.000000,5000.000000,5000.0,5000.000000
mean,173.0,554.0,4.300990e+08,2024-12-20 18:09:47.520000,2025-02-11 19:33:36,8061.782227,2025-02-04 03:22:33.600000,6491.948242,3.058003e+06,NaT,...,0.001600,0.00200,0.006400,0.006000,1.0,2953.604248,2839.921143,2959.299561,0.0,331.212097
min,173.0,554.0,3.791030e+08,2024-07-01 00:00:00,2024-07-15 00:00:00,14.160000,2024-07-23 00:00:00,0.000000,2.664679e+06,NaT,...,0.000000,0.00000,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.0,0.000000
25%,173.0,554.0,4.093039e+08,2024-09-17 00:00:00,2024-11-07 00:00:00,1145.889984,2024-11-05 00:00:00,669.862503,2.669802e+06,NaT,...,0.000000,0.00000,0.000000,0.000000,1.0,415.362740,293.033257,418.502197,0.0,303.030304
50%,173.0,554.0,4.301174e+08,2024-12-11 00:00:00,2025-02-04 00:00:00,3924.015015,2025-01-27 00:00:00,2926.585083,2.680196e+06,NaT,...,0.000000,0.00000,0.000000,0.000000,1.0,1434.978394,1280.242920,1442.731201,0.0,303.030304
75%,173.0,554.0,4.545036e+08,2025-03-27 00:00:00,2025-05-22 00:00:00,11591.912598,2025-05-08 06:00:00,9545.760254,3.304510e+06,NaT,...,0.000000,0.00000,0.000000,0.000000,1.0,4249.217651,4175.820068,4251.101562,0.0,303.030304
max,173.0,554.0,4.759011e+08,2025-06-27 00:00:00,2025-12-03 00:00:00,27260.947266,2025-08-12 00:00:00,22859.605469,4.256572e+06,NaT,...,1.000000,1.00000,1.000000,1.000000,1.0,10000.000000,10000.000000,10000.000000,0.0,10000.000000
std,0.0,0.0,2.819879e+07,NaN,NaN,9060.878906,NaN,7626.401855,6.316892e+05,NaN,...,0.039971,0.04468,0.079749,0.077236,0.0,3325.484863,3336.197510,3324.166748,0.0,251.377045



 Estatísticas GM


...

,id_grupo,id_beneficiario,Numero_do_boleto,data_inclusao,data_vencto,vl_boleto,dt_pagto,vl_pagto,id_pagador,pagador_dt_ultimo_acesso,...,pagador_cidade_assis,pagador_cidade_µGUAS FRIAS,pagador_cidade_µGUAS MORNAS,tipo_juros_M,tipo_juros_S,vl_boleto_norm,vl_pagto_norm,juros_norm,valor_abatimento_norm,qtd_acessos_pagador_norm
count,5000.0,5000.0,5.000000e+03,5000,5000,5000.000000,1604,5000.000000,5.000000e+03,0,...,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.0,5000.000000
mean,112.0,366.0,4.330546e+08,2024-12-31 20:21:41.760000,2025-12-11 07:24:40.320000,1268.836060,2025-03-25 15:22:53.566084864,351.209015,2.017058e+07,NaT,...,0.000200,0.000200,0.000200,0.999200,0.000800,3897.929932,2237.418701,3968.863281,0.0,253.319153
min,112.0,366.0,3.792501e+08,2024-07-01 00:00:00,2024-07-27 00:00:00,80.000000,2024-07-23 00:00:00,0.000000,1.837809e+07,NaT,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
25%,112.0,366.0,4.129624e+08,2024-09-30 00:00:00,2025-06-05 00:00:00,617.972504,2025-01-10 00:00:00,0.000000,1.979491e+07,NaT,...,0.000000,0.000000,0.000000,1.000000,0.000000,1763.892731,0.000000,1926.985229,0.0,0.000000
50%,112.0,366.0,4.340801e+08,2024-12-27 00:00:00,2025-11-10 00:00:00,970.000000,2025-04-08 00:00:00,0.000000,2.028464e+07,NaT,...,0.000000,0.000000,0.000000,1.000000,0.000000,2918.113037,0.000000,2980.805420,0.0,212.765961
75%,112.0,366.0,4.571390e+08,2025-04-08 00:00:00,2026-06-10 00:00:00,1622.750000,2025-06-16 00:00:00,627.882507,2.073946e+07,NaT,...,0.000000,0.000000,0.000000,1.000000,0.000000,5058.335571,4000.000000,5156.191284,0.0,212.765961
max,112.0,366.0,4.762897e+08,2025-06-30 00:00:00,2028-05-10 00:00:00,3129.916260,2025-08-12 00:00:00,1569.706299,2.113348e+07,NaT,...,1.000000,1.000000,1.000000,1.000000,1.000000,10000.000000,10000.000000,10000.000000,0.0,10000.000000
std,0.0,0.0,2.778931e+07,NaN,NaN,889.224976,NaN,574.417114,7.218347e+05,NaN,...,0.014142,0.014143,0.014143,0.028277,0.028277,2915.563965,3659.313965,2817.795898,0.0,418.146942



 Estatísticas GV


,id_grupo,id_beneficiario,Numero_do_boleto,data_inclusao,data_vencto,vl_boleto,dt_pagto,vl_pagto,id_pagador,pagador_dt_ultimo_acesso,...,status_boleto_PROTESTADO,status_boleto_REGISTRADO,banco_BANCO BRADESCO SA,pagador_cidade_Desconhecido,tipo_juros_P,vl_boleto_norm,vl_pagto_norm,juros_norm,valor_abatimento_norm,qtd_acessos_pagador_norm
count,5000.0,5000.0,5.000000e+03,5000,5000,5000.000000,4648,5000.000000,5.000000e+03,0,...,5000.000000,5000.00000,5000.0,5000.0,5000.0,5000.000000,5000.000000,5000.0,5000.000000,5000.000000
mean,67.0,213.0,4.360459e+08,2025-01-10 05:06:25.920000,2025-02-10 18:51:33.120000,642.953796,2025-02-11 07:43:28.605852160,597.390503,2.804971e+06,NaT,...,0.000800,0.00640,1.0,1.0,1.0,3826.448730,3685.207275,0.0,7.665730,78.422356
min,67.0,213.0,3.793397e+08,2024-07-01 00:00:00,2024-07-03 00:00:00,29.000000,2024-07-04 00:00:00,0.000000,4.161110e+05,NaT,...,0.000000,0.00000,1.0,1.0,1.0,0.000000,0.000000,0.0,0.000000,0.000000
25%,67.0,213.0,4.157880e+08,2024-10-11 00:00:00,2024-11-14 00:00:00,315.000000,2024-11-18 00:00:00,276.000000,1.136693e+06,NaT,...,0.000000,0.00000,1.0,1.0,1.0,1782.486694,1702.600220,0.0,0.000000,3.537319
50%,67.0,213.0,4.350066e+08,2025-01-02 00:00:00,2025-01-29 00:00:00,518.000000,2025-01-29 00:00:00,483.389999,1.149742e+06,NaT,...,0.000000,0.00000,1.0,1.0,1.0,3047.678467,2981.956177,0.0,0.000000,14.149275
75%,67.0,213.0,4.598670e+08,2025-04-17 00:00:00,2025-05-19 00:00:00,842.400009,2025-05-20 00:00:00,814.020004,4.055447e+06,NaT,...,0.000000,0.00000,1.0,1.0,1.0,5069.492065,5021.559937,0.0,0.000000,70.746376
max,67.0,213.0,4.764664e+08,2025-06-30 00:00:00,2025-11-10 00:00:00,1633.500000,2025-08-12 00:00:00,1621.050049,8.433578e+06,NaT,...,1.000000,1.00000,1.0,1.0,1.0,10000.000000,10000.000000,0.0,10000.000000,10000.000000
std,0.0,0.0,2.580995e+07,NaN,NaN,427.062683,NaN,439.828064,2.799786e+06,NaN,...,0.028277,0.07975,0.0,0.0,0.0,2661.656738,2713.231934,0.0,193.828827,382.418640



 Estatísticas GW


,id_grupo,id_beneficiario,Numero_do_boleto,data_inclusao,data_vencto,vl_boleto,dt_pagto,vl_pagto,id_pagador,pagador_dt_ultimo_acesso,...,pagador_cidade_XANXERE,pagador_cidade_Xaxim,pagador_cidade_Xique-Xique,pagador_cidade_ilheus,tipo_juros_M,vl_boleto_norm,vl_pagto_norm,juros_norm,valor_abatimento_norm,qtd_acessos_pagador_norm
count,5000.0,5000.0,5.000000e+03,5000,5000,5000.000000,2113,5000.000000,5.000000e+03,0,...,5000.000000,5000.000000,5000.000000,5000.00000,5000.0,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,212.0,658.0,4.368123e+08,2025-01-15 01:39:21.600000,2025-08-18 21:12:05.760000,4843.727539,2025-03-19 18:58:46.739233280,1332.581665,7.814915e+06,NaT,...,0.000200,0.000200,0.005400,0.00100,1.0,3123.936523,2457.603516,3130.350342,2.249791,151.407394
min,212.0,658.0,3.802175e+08,2024-07-03 00:00:00,2024-07-26 00:00:00,4.890000,2024-07-29 00:00:00,0.000000,3.231305e+06,NaT,...,0.000000,0.000000,0.000000,0.00000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000
25%,212.0,658.0,4.178916e+08,2024-10-18 00:00:00,2025-03-20 00:00:00,1161.660034,2025-01-09 00:00:00,0.000000,3.273850e+06,NaT,...,0.000000,0.000000,0.000000,0.00000,1.0,746.806702,0.000000,755.082275,0.000000,0.000000
50%,212.0,658.0,4.413524e+08,2025-01-29 00:00:00,2025-07-14 00:00:00,2773.109985,2025-04-01 00:00:00,0.000000,3.699559e+06,NaT,...,0.000000,0.000000,0.000000,0.00000,1.0,1787.153137,0.000000,1800.580933,0.000000,0.000000
75%,212.0,658.0,4.572205e+08,2025-04-08 00:00:00,2025-12-08 00:00:00,6894.772583,2025-06-10 00:00:00,2168.912415,1.281845e+07,NaT,...,0.000000,0.000000,0.000000,0.00000,1.0,4448.083984,4000.000061,4453.049316,0.000000,0.000000
max,212.0,658.0,4.764020e+08,2025-06-30 00:00:00,2029-10-09 00:00:00,15494.441406,2025-08-12 00:00:00,5422.281250,2.113568e+07,NaT,...,1.000000,1.000000,1.000000,1.00000,1.0,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
std,0.0,0.0,2.617111e+07,NaN,NaN,4922.048340,NaN,2006.524170,6.671667e+06,NaN,...,0.014143,0.014142,0.073293,0.03161,0.0,3177.666992,3700.611816,3178.188232,142.240448,484.431885
